# ML-10 — Content Action Playbook

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vinukondablessena/flyrank-ml-internship/blob/main/work/notebooks/w07_action_playbook.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Ranked actions + reason codes

*The queue: what to do first, and why, in words a human trusts.*



The action queue ranks content by a baseline decision-support score and assigns a reason code that explains why an item should be reviewed. The reason codes are intended to make the ranking understandable to a human reviewer rather than treating the score as an automatic decision.

The main actions are:

* **REFRESH** — review content that shows signs of declining performance or outdatedness.
* **REVIEW_POSITION** — review content with weaker search position where improvement may be possible.
* **REVIEW_ENGAGEMENT** — review content with relatively weak engagement signals.
* **MONITOR** — keep the item under observation when the evidence is weaker or mixed.

The queue is a prioritization tool. A high-ranked item means it should be reviewed earlier; it does not mean that the recommended action should automatically be performed.


In [14]:
import os

print("Current folder:", os.getcwd())

if not os.path.exists("/content/flyrank-ml-internship"):
    !git clone https://github.com/vinukondablessena/flyrank-ml-internship.git

%cd /content/flyrank-ml-internship

print("\nRepository ready.")
print(os.listdir(".")[:20])

Current folder: /content/flyrank-ml-internship
/content/flyrank-ml-internship

Repository ready.
['work', 'outputs', 'README.md', '.github', '.gitignore', 'docs', 'data', 'SETUP.md', 'scripts', 'LICENSE', 'submission', 'DATA_USE.md', 'GUIDE.md', 'skills', 'requirements.txt', 'AGENTS.md', '.git', 'notebooks', 'CLAUDE.md']


In [15]:
%cd /content/flyrank-ml-internship

import os

print("Repository contents:")
for root, dirs, files in os.walk("/content/flyrank-ml-internship"):
    level = root.replace("/content/flyrank-ml-internship", "").count(os.sep)
    indent = "  " * level
    print(f"{indent}{os.path.basename(root)}/")
    for file in files[:20]:
        print(f"{indent}  {file}")

/content/flyrank-ml-internship
Repository contents:
flyrank-ml-internship/
  README.md
  .gitignore
  SETUP.md
  LICENSE
  DATA_USE.md
  GUIDE.md
  requirements.txt
  AGENTS.md
  CLAUDE.md
  work/
    README.md
    capstone_report_template.md
    outputs/
      baseline_action_score.csv
      action_playbook_metrics.json
    notebooks/
      w03_data_contract.ipynb
      capstone.ipynb
      w04_baseline_score.ipynb
      w04_signal_audit.ipynb
      w05_model.ipynb
      w03_feature_leakage_check.ipynb
      w07_action_playbook.ipynb
      w01_research_question.ipynb
      w06_validation_audit.ipynb
      w02_ml_task_framing.ipynb
  outputs/
    model_report.md
    refresh_queue_sample.csv
    charts/
      top_reason_codes.svg
      confidence_mix.svg
      top_feature_importance.svg
      action_mix.svg
      trend_distribution.svg
  .github/
    workflows/
      personalize.yml
      smoke-test.yml
      data-path-smoke.yml
  docs/
    intern-free-tooling-guide.md
    data-dictiona

In [16]:

%cd /content

!rm -rf flyrank-ml-internship

!git clone --depth 1 https://github.com/vinukondablessena/flyrank-ml-internship.git

%cd /content/flyrank-ml-internship

print("Repository contents:")
!ls -la


/content
Cloning into 'flyrank-ml-internship'...
remote: Enumerating objects: 91, done.
remote: Counting objects: 100% (91/91), done.
remote: Compressing objects: 100% (71/71), done.
remote: Total 91 (delta 12), reused 63 (delta 4), pack-reused 0 (from 0)
Receiving objects: 100% (91/91), 1.87 MiB | 13.50 MiB/s, done.
Resolving deltas: 100% (12/12), done.
/content/flyrank-ml-internship
Repository contents:
total 104
drwxr-xr-x 12 root root  4096 Aug 20 14:03 .
drwxr-xr-x  1 root root  4096 Aug 20 14:03 ..
-rw-r--r--  1 root root   654 Aug 20 14:03 AGENTS.md
-rw-r--r--  1 root root   654 Aug 20 14:03 CLAUDE.md
drwxr-xr-x  3 root root  4096 Aug 20 14:03 data
-rw-r--r--  1 root root  2763 Aug 20 14:03 DATA_USE.md
drwxr-xr-x  2 root root  4096 Aug 20 14:03 docs
drwxr-xr-x  8 root root  4096 Aug 20 14:03 .git
drwxr-xr-x  3 root root  4096 Aug 20 14:03 .github
-rw-r--r--  1 root root   993 Aug 20 14:03 .gitignore
-rw-r--r--  1 root root 10408 Aug 20 14:03 GUIDE.md
-rw-r--r--  1 root root  128

In [17]:
!python scripts/01_prepare_features.py

Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


In [18]:
%cd /content/flyrank-ml-internship

!python scripts/01_prepare_features.py

/content/flyrank-ml-internship
Prepared 30,000 rows from 30,000 raw rows
Wrote /content/flyrank-ml-internship/data/processed/refresh_feature_vector.csv


In [19]:
# =========================================================
# ML-10 — SECTION 1: RANKED ACTIONS + REASON CODES
# =========================================================

import os
import pandas as pd
import numpy as np

REPO = "/content/flyrank-ml-internship"

DATA_PATH = os.path.join(
    REPO,
    "data/processed/refresh_feature_vector.csv"
)

OUTPUT_DIR = os.path.join(
    REPO,
    "work/outputs"
)

os.makedirs(OUTPUT_DIR, exist_ok=True)

# Load prepared data
df = pd.read_csv(DATA_PATH)

print("Dataset shape:", df.shape)

# ---------------------------------------------------------
# Baseline action score
# ---------------------------------------------------------

df["baseline_score"] = (
    (1 - df["ctr"].fillna(df["ctr"].median())) * 0.35
    + (df["avg_position"].fillna(df["avg_position"].median())
       / df["avg_position"].fillna(df["avg_position"].median()).max()) * 0.25
    + (1 - df["engagement_rate"].fillna(
        df["engagement_rate"].median()
    )) * 0.20
    + (df["days_since_last_update"].fillna(
        df["days_since_last_update"].median()
    ) / df["days_since_last_update"].fillna(
        df["days_since_last_update"].median()
    ).max()) * 0.20
)

# Rank highest priority first
df = df.sort_values(
    "baseline_score",
    ascending=False
).reset_index(drop=True)

df["rank"] = np.arange(1, len(df) + 1)

# ---------------------------------------------------------
# Reason codes
# ---------------------------------------------------------

ctr_median = df["ctr"].median()

def assign_reason(row):
    if row["days_since_last_update"] >= 180:
        return "STALE_CONTENT"
    elif row["avg_position"] >= 20:
        return "WEAK_POSITION"
    elif row["ctr"] < ctr_median:
        return "LOW_CTR"
    elif row["engagement_rate"] < df["engagement_rate"].median():
        return "LOW_ENGAGEMENT"
    else:
        return "MONITOR"

df["reason_code"] = df.apply(assign_reason, axis=1)

# ---------------------------------------------------------
# Reason → action mapping
# ---------------------------------------------------------

reason_to_action = {
    "STALE_CONTENT": "REFRESH",
    "WEAK_POSITION": "REVIEW_POSITION",
    "LOW_CTR": "REVIEW_ENGAGEMENT",
    "LOW_ENGAGEMENT": "REVIEW_ENGAGEMENT",
    "MONITOR": "MONITOR"
}

df["action"] = df["reason_code"].map(reason_to_action)

# ---------------------------------------------------------
# Ranked queue
# ---------------------------------------------------------

queue_columns = [
    "rank",
    "content_id",
    "baseline_score",
    "reason_code",
    "action",
    "impressions_90d",
    "avg_position",
    "ctr"
]

queue_columns = [
    c for c in queue_columns
    if c in df.columns
]

queue = df[queue_columns].copy()

QUEUE_PATH = os.path.join(
    OUTPUT_DIR,
    "baseline_action_score.csv"
)

queue.to_csv(
    QUEUE_PATH,
    index=False
)

print("\nQueue written to:")
print(QUEUE_PATH)

print("\nRows ranked:", len(queue))

print("\nReason-code distribution:")
print(queue["reason_code"].value_counts())

print("\nAction distribution:")
print(queue["action"].value_counts())

print("\nTop 10 ranked actions:")
display(queue.head(10))

Dataset shape: (30000, 52)

Queue written to:
/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv

Rows ranked: 30000

Reason-code distribution:
reason_code
MONITOR          11878
LOW_CTR           9378
WEAK_POSITION     8570
STALE_CONTENT      174
Name: count, dtype: int64

Action distribution:
action
MONITOR              11878
REVIEW_ENGAGEMENT     9378
REVIEW_POSITION       8570
REFRESH                174
Name: count, dtype: int64

Top 10 ranked actions:


,rank,content_id,baseline_score,reason_code,action,impressions_90d,avg_position,ctr
0,1,content_661e1745db72,0.810724,WEAK_POSITION,REVIEW_POSITION,1,245.0,0.0
1,2,content_23f1cc8851a9,0.793519,WEAK_POSITION,REVIEW_POSITION,1,184.0,0.0
2,3,content_6476d1d8c050,0.787012,STALE_CONTENT,REFRESH,304,67.8,0.0
3,4,content_7a888d3d99c8,0.786808,STALE_CONTENT,REFRESH,95,67.6,0.0
4,5,content_8d56efff1e71,0.785178,STALE_CONTENT,REFRESH,1,35.0,0.0
5,6,content_f6fdf87348f6,0.783163,STALE_CONTENT,REFRESH,2,32.5,0.0
6,7,content_15fe075b97bc,0.781370,STALE_CONTENT,REFRESH,5,67.0,0.0
7,8,content_d25a099b3726,0.779355,STALE_CONTENT,REFRESH,202,64.5,0.0
8,9,content_7275a6a3a8eb,0.774642,WEAK_POSITION,REVIEW_POSITION,2,165.5,0.0
9,10,content_71a31b831092,0.770050,WEAK_POSITION,REVIEW_POSITION,1,161.0,0.0


## 2. Intended use and limits

*Who uses this, for what — and where it stops being valid.*

The playbook is intended for content teams to use as a decision-support queue for prioritizing pages that may deserve review. A reviewer can use the rank, reason code, and supporting metrics to decide which content to inspect first.

The queue is not intended to make automatic publishing, deletion, rewriting, or SEO decisions. The baseline score is an observed prioritization signal, not a prediction of guaranteed future performance.

The main limits are that the validation was performed on a limited dataset, the client-grouped evaluation produced more conservative performance than the random split, and the decline label is a current-window proxy rather than a direct future outcome. The queue therefore supports directional prioritization, not causal conclusions or production automation.

In [20]:
# =========================================================
# 2. INTENDED USE AND LIMITS
# =========================================================

print("=== INTENDED USE CHECK ===")

print("Rows in decision-support queue:", len(queue))

print("\nTop 50 items available for human review:")
display(queue.head(50))

print("\nConservative validation reference:")
print("ROC-AUC: 0.6044")
print("Average Precision: 0.5998")
print("Precision@50: 0.72")

print("\nUse:")
print("Human-reviewed content prioritization only.")

print("\nNot for:")
print("- Automatic publishing")
print("- Automatic deletion")
print("- Automatic content rewriting")
print("- Claims of causal or future business impact")

=== INTENDED USE CHECK ===
Rows in decision-support queue: 30000

Top 50 items available for human review:


,rank,content_id,baseline_score,reason_code,action,impressions_90d,avg_position,ctr
0,1,content_661e1745db72,0.810724,WEAK_POSITION,REVIEW_POSITION,1,245.0,0.0
1,2,content_23f1cc8851a9,0.793519,WEAK_POSITION,REVIEW_POSITION,1,184.0,0.0
2,3,content_6476d1d8c050,0.787012,STALE_CONTENT,REFRESH,304,67.8,0.0
3,4,content_7a888d3d99c8,0.786808,STALE_CONTENT,REFRESH,95,67.6,0.0
4,5,content_8d56efff1e71,0.785178,STALE_CONTENT,REFRESH,1,35.0,0.0
5,6,content_f6fdf87348f6,0.783163,STALE_CONTENT,REFRESH,2,32.5,0.0
6,7,content_15fe075b97bc,0.781370,STALE_CONTENT,REFRESH,5,67.0,0.0
7,8,content_d25a099b3726,0.779355,STALE_CONTENT,REFRESH,202,64.5,0.0
8,9,content_7275a6a3a8eb,0.774642,WEAK_POSITION,REVIEW_POSITION,2,165.5,0.0
9,10,content_71a31b831092,0.770050,WEAK_POSITION,REVIEW_POSITION,1,161.0,0.0



Conservative validation reference:
ROC-AUC: 0.6044
Average Precision: 0.5998
Precision@50: 0.72

Use:
Human-reviewed content prioritization only.

Not for:
- Automatic publishing
- Automatic deletion
- Automatic content rewriting
- Claims of causal or future business impact


## 3. Human review + the no-go list

*What a person must check before acting. What should never be automated.*

Every ranked recommendation should be reviewed by a person before an action is taken. The reviewer should check the page's purpose, current content quality, search intent, business importance, recent changes, and any information that is not represented in the model features.

The reason code should be treated as a prompt for investigation, not as a final diagnosis. For example, a WEAK_POSITION item may require checking search intent and competing results before deciding whether a refresh is appropriate.

The following should not be automated by this playbook:

Publishing or deleting content.
Rewriting or changing content without human approval.
Making claims about business impact or causality.
Changing important pages solely because they have a high model score.
Applying the same action to every page with the same reason code.
Making decisions where the available data are clearly stale, incomplete, or contradictory.

The model prioritizes review; a human remains responsible for the final content decision.

In [21]:
# =========================================================
# 3. HUMAN REVIEW + NO-GO LIST
# =========================================================

print("=== HUMAN REVIEW CHECK ===")

# Count the items that require an explicit human review action
review_actions = [
    "REFRESH",
    "REVIEW_POSITION",
    "REVIEW_ENGAGEMENT"
]

review_queue = queue[
    queue["action"].isin(review_actions)
].copy()

print("Total ranked items:", len(queue))
print("Items requiring human review:", len(review_queue))
print(
    "Items marked MONITOR:",
    int((queue["action"] == "MONITOR").sum())
)

print("\nHuman-review action counts:")
print(review_queue["action"].value_counts())

print("\nTop 10 items requiring human review:")
display(review_queue.head(10))

# No-go actions are deliberately not represented
NO_GO_ACTIONS = [
    "AUTO_PUBLISH",
    "AUTO_DELETE",
    "AUTO_REWRITE",
    "AUTO_REDIRECT"
]

automated_no_go_present = [
    action for action in NO_GO_ACTIONS
    if action in queue["action"].unique()
]

print("\nAutomated no-go actions present in queue:")
print(automated_no_go_present)

if not automated_no_go_present:
    print(
        "\nPASS: No automatic publish, delete, rewrite, "
        "or redirect actions are included."
    )
else:
    print("\nREVIEW REQUIRED: Automated no-go action detected.")

=== HUMAN REVIEW CHECK ===
Total ranked items: 30000
Items requiring human review: 18122
Items marked MONITOR: 11878

Human-review action counts:
action
REVIEW_ENGAGEMENT    9378
REVIEW_POSITION      8570
REFRESH               174
Name: count, dtype: int64

Top 10 items requiring human review:


,rank,content_id,baseline_score,reason_code,action,impressions_90d,avg_position,ctr
0,1,content_661e1745db72,0.810724,WEAK_POSITION,REVIEW_POSITION,1,245.0,0.0
1,2,content_23f1cc8851a9,0.793519,WEAK_POSITION,REVIEW_POSITION,1,184.0,0.0
2,3,content_6476d1d8c050,0.787012,STALE_CONTENT,REFRESH,304,67.8,0.0
3,4,content_7a888d3d99c8,0.786808,STALE_CONTENT,REFRESH,95,67.6,0.0
4,5,content_8d56efff1e71,0.785178,STALE_CONTENT,REFRESH,1,35.0,0.0
5,6,content_f6fdf87348f6,0.783163,STALE_CONTENT,REFRESH,2,32.5,0.0
6,7,content_15fe075b97bc,0.781370,STALE_CONTENT,REFRESH,5,67.0,0.0
7,8,content_d25a099b3726,0.779355,STALE_CONTENT,REFRESH,202,64.5,0.0
8,9,content_7275a6a3a8eb,0.774642,WEAK_POSITION,REVIEW_POSITION,2,165.5,0.0
9,10,content_71a31b831092,0.770050,WEAK_POSITION,REVIEW_POSITION,1,161.0,0.0



Automated no-go actions present in queue:
[]

PASS: No automatic publish, delete, rewrite, or redirect actions are included.


## 4. Monitoring / retrain triggers

*What would tell you the recommendations went stale?*

The recommendations should be monitored for signs that the data or model has become stale. Monitoring should focus on changes in the input distributions, the ranking behavior, and the usefulness of the recommendations when reviewed by humans.

A review should be triggered if:

Key traffic, CTR, position, or engagement distributions change materially from the development data.
The proportion of items receiving each reason code changes substantially.
The ranked queue becomes dominated by one reason code without a clear business explanation.
New clients, content types, or traffic sources appear that were not represented during development.
Human reviewers repeatedly disagree with the recommended reason or action.
Observed validation performance on newly available labeled data falls materially below the measured client-grouped result.

Retraining should be considered after a sustained distribution or performance change rather than after a single unusual observation. Any retrained model should be re-evaluated using a grouped or time-aware validation design before being used for decision support again.

In [22]:
# =========================================================
# 4. MONITORING / RETRAIN TRIGGERS
# =========================================================

print("=== MONITORING / RETRAIN CHECK ===")

# Current queue distribution
reason_distribution = (
    queue["reason_code"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

action_distribution = (
    queue["action"]
    .value_counts(normalize=True)
    .mul(100)
    .round(2)
)

print("\nCurrent reason-code distribution (%):")
print(reason_distribution)

print("\nCurrent action distribution (%):")
print(action_distribution)

# Development reference metrics from ML-09
REFERENCE_AUC = 0.6044
REFERENCE_AP = 0.5998
REFERENCE_P50 = 0.72

print("\nReference client-grouped validation:")
print("ROC-AUC:", REFERENCE_AUC)
print("Average Precision:", REFERENCE_AP)
print("Precision@50:", REFERENCE_P50)

print("\nMonitoring triggers:")
print("1. Check feature/data distributions for material drift.")
print("2. Check reason-code and action distributions.")
print("3. Review new clients/content types not represented in development.")
print("4. Track human reviewer disagreement.")
print("5. Revalidate if new labeled outcomes become available.")
print("6. Consider retraining after sustained drift or performance decline.")

print("\nRetraining rule:")
print(
    "Retrain only after sustained evidence of data or performance "
    "change, followed by grouped or time-aware revalidation."
)

=== MONITORING / RETRAIN CHECK ===

Current reason-code distribution (%):
reason_code
MONITOR          39.59
LOW_CTR          31.26
WEAK_POSITION    28.57
STALE_CONTENT     0.58
Name: proportion, dtype: float64

Current action distribution (%):
action
MONITOR              39.59
REVIEW_ENGAGEMENT    31.26
REVIEW_POSITION      28.57
REFRESH               0.58
Name: proportion, dtype: float64

Reference client-grouped validation:
ROC-AUC: 0.6044
Average Precision: 0.5998
Precision@50: 0.72

Monitoring triggers:
1. Check feature/data distributions for material drift.
2. Check reason-code and action distributions.
3. Review new clients/content types not represented in development.
4. Track human reviewer disagreement.
5. Revalidate if new labeled outcomes become available.
6. Consider retraining after sustained drift or performance decline.

Retraining rule:
Retrain only after sustained evidence of data or performance change, followed by grouped or time-aware revalidation.


## 5. Exports for the paper

*Write the queue (and any figures you want to reuse) to work/outputs/ — your paper builds on these files.*



The ranked action queue is exported to `work/outputs/` so that the paper can reuse the same decision-support output without manually recreating it.

The main export is `baseline_action_score.csv`, containing the ranked content items, baseline score, reason code, recommended review action, and supporting metrics.

The export is a reproducible output of this notebook. It is intended for analysis and paper development, not as a production decision system. The queue itself remains outside version control according to the repository's data-handling rules.


In [23]:
# =========================================================
# 5. EXPORTS FOR THE PAPER
# =========================================================

import os
import json

OUTPUT_DIR = "/content/flyrank-ml-internship/work/outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# ---------------------------------------------------------
# 1. Export ranked queue
# ---------------------------------------------------------

queue_path = os.path.join(
    OUTPUT_DIR,
    "baseline_action_score.csv"
)

queue.to_csv(
    queue_path,
    index=False
)

# ---------------------------------------------------------
# 2. Export summary metrics
# ---------------------------------------------------------

summary = {
    "rows_ranked": int(len(queue)),
    "reason_code_counts": {
        str(k): int(v)
        for k, v in queue["reason_code"].value_counts().items()
    },
    "action_counts": {
        str(k): int(v)
        for k, v in queue["action"].value_counts().items()
    },
    "validation": {
        "split": "client-grouped",
        "roc_auc": 0.6044,
        "average_precision": 0.5998,
        "precision_at_50": 0.72
    },
    "intended_use": "human-reviewed decision support",
    "automatic_actions": False
}

metrics_path = os.path.join(
    OUTPUT_DIR,
    "action_playbook_metrics.json"
)

with open(metrics_path, "w") as f:
    json.dump(summary, f, indent=2)

# ---------------------------------------------------------
# 3. Verify exports
# ---------------------------------------------------------

print("=== EXPORTS ===")

print("\nQueue export:")
print(queue_path)
print("Exists:", os.path.exists(queue_path))

print("\nMetrics export:")
print(metrics_path)
print("Exists:", os.path.exists(metrics_path))

print("\nQueue rows:", len(queue))

print("\nOutput directory contents:")
for filename in sorted(os.listdir(OUTPUT_DIR)):
    print("-", filename)

=== EXPORTS ===

Queue export:
/content/flyrank-ml-internship/work/outputs/baseline_action_score.csv
Exists: True

Metrics export:
/content/flyrank-ml-internship/work/outputs/action_playbook_metrics.json
Exists: True

Queue rows: 30000

Output directory contents:
- action_playbook_metrics.json
- baseline_action_score.csv


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.